## Task 3, part 3 - Modelling of the second model using the protein data

This model predicts each perturbation's RNA log2FC fingerprint from that same perturbation's own effect on the 24 (20 real protein readouts + 4 isotype controls) measured surface proteins.

In [1]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [2]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()

selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

## Compute the protein log2FC "fingerprint"

Mirrors exactly what Task3_01 did for RNA: for each condition, compare the mean protein expression of each perturbation's cells to that condition's control cells, on a log2 scale with a pseudocount. 4 of the 24 measured "proteins" are isotype controls (antibody background-binding controls, not real markers) and are dropped, leaving us with 20 real surface markers.

In [3]:
protein = sc.read_h5ad(f"{DATA_DIR}/protein_qc_filtered.h5ad")

# drop isotype controls (here isotype_control in var is nan since they are the control themselfes)
protein = protein[:, protein.var['Isotype_control']!='nan'].copy()

# keep only control cells and cells belonging to one of the 50 selected perturbations
relevant_mask = protein.obs["perturbation"].isin(selected_50) | (protein.obs["perturbation"] == "control")
protein = protein[relevant_mask.values].copy()

# normalize the protein expression data 
sc.pp.normalize_total(protein, target_sum=1e4)
protein_norm = np.asarray(protein.X.todense())

protein.shape

/tmp/ipykernel_62464/1157936093.py:11: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(protein, target_sum=1e4)


(92532, 20)

In [4]:
# compute the mean protein expression across all control cells under each condition
control_means_protein = {}
for cond in conditions:
    mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == "control")
    control_means_protein[cond] = protein_norm[mask.values].mean(axis=0)

# log2FC protein vector of 20 elements per perturbation and condition combination, relative to control cells of that condition
protein_FC = {}
# iterate over all conditions and perturbations
for pert in selected_50:
    for cond in conditions:
        # filter for all cells that have that condition and perturbation and average the protein expression across these cells
        mask = (protein.obs["perturbation_2"] == cond) & (protein.obs["perturbation"] == pert)
        pert_mean = protein_norm[mask.values].mean(axis=0)
        # calculate the log2FC with pseudocount for the protein expression in perturbed cells compared to control cells in the same condition
        protein_FC[(pert, cond)] = np.log2((pert_mean + 1) / (control_means_protein[cond] + 1))

# define two level index from the two keys for each FC vector (condition and perturbation)
protein_FC_index = pd.MultiIndex.from_tuples(protein_FC.keys(), names=["perturbation", "condition"])
# convert to dataframe with index = perturbation, condition and columnames = proteins
protein_FC_df = pd.DataFrame(np.vstack(list(protein_FC.values())), index=protein_FC_index, columns=protein.var_names)

protein_FC_df.shape

(150, 20)

## Reduce the target: PCA on the training RNA FC vectors

The RNA fingerprint we want to predict has 2042 values, but we only have 20 protein features and 40 training perturbations per condition. Predicting 2042 outputs from 40 examples would almost certainly overfit, so we compress the target first.

For each condition we fit a PCA on the RNA FC vectors of the 40 training perturbations and keep the first 10 components. Instead of predicting all 2042 genes, the model then only has to predict the 10 PC scores, which are then transformed back into a full FC vector. The PCA is fitted on the training perturbations only, so the 10 held-out perturbations never influence the PCA axes.

In [ ]:
n_PCs = 10

# per condition fit PCA on the 40 training genes' RNA FC vectors, and store each training gene's score
target_pca_by_condition = {}
target_scores = {}  # store the scores of the 10 PCs for each training pair of perturbation and condition
for cond in conditions: # iterate over all conditions
    train_fingerprints = np.vstack([pert_FC_selected.loc[(g, cond)].values for g in train_40]) # store RNA FC vectors for each perturbation under the condition in an array (40, 2042)
    pca = PCA(n_components= n_PCs, random_state=42) # create PCA object that keeps 10 components and has a seed for reproducibility
    scores = pca.fit_transform(train_fingerprints) # fit the pca on all training fingerprints and store one row of 10 scores (for all components) for each perturbation in training
    target_pca_by_condition[cond] = pca # store the fitted PCA 
    for gene, score in zip(train_40, scores): # pair each gene name in train 40 and the scores (along the 10 pca components) and iterate over them
        target_scores[(gene, cond)] = score # store the scores in the target_scores dictionary with two keys

# compute how much of the training FC vectors' variance these 10 PCs capture, per condition
{cond: target_pca_by_condition[cond].explained_variance_ratio_.sum() for cond in conditions}

{'Control': np.float32(0.61914194),
 'IFNγ': np.float32(0.6695988),
 'Co-culture': np.float32(0.76921785)}

## Ridge regression prediction

In this step we define a ridge regression to predict the log2FC vector across all 2042 genes for a given perturbation under a condition given a pool of training genes and a regularization strength of alpha (the query is always excluded from the pool so that we can use this function for LOOCV). The input for the regression are the protein FC values across all 20 proteins which are Z-transformed before fitting and the target is defined by the 10 PC scores. After fitting this relationship we can predict the PC scores for an unknown perturbation and transform it back into the full FC vector using the PCA for that condition.

In [6]:
# define a function to predict the gene expression log2FC vector from the protein measurements under a given perturbation in one condition
def ridge_predict(query_gene, condition, alpha, pool_genes): # alpha will need to be selected later, pool genes = training perturbations
    """Predict an RNA fingerprint via Ridge regression from protein log2FC to RNA target-PC scores."""
    # exclude the query gene itself from the pool used to fit the model
    fit_genes = [g for g in pool_genes if g != query_gene] # define all genes used to fit the regression as those which are in the pool but not in the query

    # define input (protein data for a perturbation and condition) and target (scores of the 10 PCs for a perturbation and condition) of the model
    X = np.vstack([protein_FC_df.loc[(g, condition)].values for g in fit_genes]) # store for all perturbation the protein data in an array
    y = np.vstack([target_scores[(g, condition)] for g in fit_genes]) # store for all perturbation the target scores in an array 

    # z-transform the protein features before regularized regression so there is no difference in penalization purely based on the scale of the data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X) # scaler learns and stores the mean and sd of the data used for the transformation so it can be used later to transform the query data and it transforms the X at the same time

    ridge = Ridge(alpha=alpha) # creates a ridge regression model with regularization strenght alpha
    ridge.fit(X_scaled, y) # fit the model on the scaled protein data to model the 10 PC scores of the RNA data 

    query_X = scaler.transform(protein_FC_df.loc[(query_gene, condition)].values.reshape(1, 20)) # grab the protein data (only values) of the query perturbation under the condition and turn it into a 1 x 20 matrix (instead of a normal vector) and scale it according to the pool mean and sd
    pred_scores = ridge.predict(query_X) # return the predicted value of each of teh 10 PCs for the perturbation

    # reconstruct the full RNA fingerprint from the predicted target-PC scores
    pred_fingerprint = target_pca_by_condition[condition].inverse_transform(pred_scores)[0] # transform the 10 PC scores back into the full FC vector by usiing the condition's PCA
    return pred_fingerprint # return the fingerprint(log2FC vector for RNA expression)

## Choosing the Ridge regularization strength (alpha) via leave-one-out cross-validation

Hold out one training gene at a time, predict it from the other 39 (per condition), and compare a few candidate alpha values for the model performance (MSE).

In [7]:
candidate_alphas = [0.1, 1, 10, 100, 1000, 10_000, 100_000, 1_000_000]

cv_mse_by_alpha = {} # create empty dictionary for mse values for each tested alpha computed by LOOCV
for alpha in candidate_alphas: # iterate over each alpha 
    squared_errors = [] # create empty list of squared errors used for MSE computation
    for cond in conditions: # iterate over conditions
        for gene in train_40: # iterate over training genes 
            # ridge_predict excludes the query gene itself from the pool, so this is a leave-one-out prediction
            pred = ridge_predict(gene, cond, alpha, train_40) # predict the left out perturbation's FC vector
            true = pert_FC_selected.loc[(gene, cond)].values # extract the actual left out perturbation's FC vector
            squared_errors.append(np.mean((true - pred) ** 2)) # append the square error to the list of squared errors 
    cv_mse_by_alpha[alpha] = np.mean(squared_errors) # append the MSE for each alpha to the mse by alpha dictionary (averaged over conditions and perturbations so over 120 = 3 x 40 entries)

best_alpha = min(cv_mse_by_alpha, key=cv_mse_by_alpha.get) # return the best alpha i.e. the regularization parameter that returnss the minimum MSE (we use the key argument again so we get the alpha corresponding to the minimum MSE not the smallest alpha)
cv_mse_by_alpha, best_alpha

({0.1: np.float32(0.007245487),
  1: np.float32(0.005463494),
  10: np.float32(0.0033818756),
  100: np.float32(0.002320617),
  1000: np.float32(0.0021854814),
  10000: np.float32(0.0021873685),
  100000: np.float32(0.0021882078),
  1000000: np.float32(0.0021883)},
 1000)

## Predict test perturbations effects on the RNA expression FC vectors and evaluate the model 

Use the chosen alpha to predict each of the 10 held-out genes' RNA fingerprint from their own protein log2FC, then evaluate with the same metrics used for the baseline model so the results are directly comparable.

In [8]:
# for each test perturbation predict the log2FC vectors under each condition using the alpha found in the previou part
ridge_predictions = {
    (gene, cond): ridge_predict(gene, cond, best_alpha, train_40) # store the predictions for each perturbation/condition pair in a dictionary
    for cond in conditions # iterate over all three conditions
    for gene in test_10 # iterate over all 10 test perturbations
}

def evaluate_predictions(true_df, predictions_by_row): # define evaluation function for predictions from model 2 
    records = [] # create empty list
    for (pert, cond), true_fc in true_df.iterrows(): # iterrate over the true FC vectors for each prediction/condition pair 
        pred_fc = predictions_by_row[(pert, cond)] # pull prediction row corresponding to the pertubation/condition pair 
        pearson_r, _ = pearsonr(true_fc, pred_fc) # compute pearson correlation and store correlation value but drop the p value
        spearman_r, _ = spearmanr(true_fc, pred_fc) # same for the spearman correlation and p value
        mse = np.mean((true_fc - pred_fc) ** 2) # compute mean squared error across all genes in the FC vector 
        records.append({
            "perturbation": pert,
            "condition": cond,
            "pearson_r": pearson_r,
            "spearman_r": spearman_r,
            "mse": mse,
        }) # append to the records list for each perturbation/condition pair the correlation values and mse 
    return pd.DataFrame(records) # return the records in the form of a dataframe


ridge_eval = evaluate_predictions(pert_FC_selected.loc[test_10, :], ridge_predictions) # call the evaluation function on the true FC vectors from the selected perturbations and the predictions from the ridge regression 
ridge_eval

,perturbation,condition,pearson_r,spearman_r,mse
0,KCNN4,Control,0.828076,0.452951,0.001510
1,KCNN4,IFNγ,0.844564,0.393301,0.001101
2,KCNN4,Co-culture,0.864985,0.340053,0.001128
3,TIMM50,Control,0.740555,0.392573,0.002776
4,TIMM50,IFNγ,0.626292,0.295205,0.003330
5,TIMM50,Co-culture,0.700670,0.198601,0.003849
6,TXNDC17,Control,0.807620,0.494533,0.004335
7,TXNDC17,IFNγ,0.623327,0.438919,0.004303
8,TXNDC17,Co-culture,0.753070,0.366474,0.004496
9,CORO1A,Control,0.804729,0.374306,0.001203


In [9]:
metrics = ["pearson_r", "spearman_r", "mse"] # define the relevant metrics to extract

# model performance in the metrics, averaged for each condition
per_condition = ridge_eval.groupby("condition")[metrics].agg(["mean", "std"])

# model performance in the metrics but averaged across all test perturbations and conditions 
overall = ridge_eval[metrics].agg(["mean", "std"])

per_condition

pearson_r           spearman_r                 mse          
                mean       std       mean       std      mean       std
condition                                                              
Co-culture  0.593808  0.498049   0.265731  0.106393  0.003967  0.005928
Control     0.755588  0.135749   0.399165  0.116829  0.002491  0.001575
IFNγ        0.719868  0.250896   0.377632  0.097377  0.002411  0.001889

In [10]:
overall

,pearson_r,spearman_r,mse
mean,0.689767,0.347582,0.002956
std,0.327519,0.118914,0.003651


## Discussion
The model uses each perturbation's 20 surface-protein log2FC vector which is build exactly like the one for the RNA expression values and aims to predict the RNA log2FC vector compressed to 10 PCs using a PCA fit on the training perturbations. The learning algorithm we used was a ridge regression with an alpha chosen by LOOCV over the training perturbations. Via inverse-PCA the full log2FC vectors are retrieved from the predicted 10 PCs.

The model performance was measured using MSE as well as Pearson and Spearman correlation as we have used before for the baslione model. With a Pearson correlation of 0.690 vs 0.688, a spearman correlation of 0.348 vs 0.349, and an MSE of 0.00296 vs 0.00297 the model seems to perform just as well as the baseline model and does clearly not outperform it. Also the differences across the different condiotions behave similarly with Control performing best and Co-culture worst, which is also the most variable. This is also concordant with the structure of the model itself where the inverse transformation returns the mean log2FC vector + the corrections across the 10 PCs which via a high alpha (1000 in this case) are regularized strongly and thus shrink towards 0. This means the model is very similar to the mean-based baseline model. The fact there is a real CV minimum tells us that there is some real link between the protein data and the transcriptional response howerver this is very weak, possibly because the protein-panel is very small and most perturbations do not influence the expression of these surface proteins which mainly relate to immune-visibility and antigen-presentation. Also the 10 PCs capture only a part of the training variance (62-77%) under each condition so not all variance can be reconstructed from the predicted scores exactly 

Possibly more perturbations to train from could solve this problem as well as richer features which comprise a more diverse panel of proteins or also additional knowledge on the connection between genes and proteins could increase the model performance. However, overall it has been shown in the past that most complex models do not yet outperform very simple baseline models which predict the mean across perturbations (model 1) as shown by Constantin Ahlmann-Eltze, Wolfgang Huber & Simon Anders in 2025. 